# **Custom Prompts in CrewAI**

We can perform low-level prompt customization in CrewAI, enabling super custom and complex use cases for different models and languages.


## Why Customize Prompts?

Although CrewAI’s default prompts work well for many scenarios, low-level customization opens the door to significantly more flexible and powerful agent behavior.

Here’s why you might want to take advantage of this deeper control:

**Optimize for specific LLMs** – Different models (such as GPT-4, Claude, or Llama) thrive with prompt formats tailored to their unique architectures.

**Change the language** – Build agents that operate exclusively in languages beyond English, handling nuances with precision.

**Specialize for complex domains** – Adapt prompts for highly specialized industries like healthcare, finance, or legal.

**Adjust tone and style** – Make agents more formal, casual, creative, or analytical.
Support super custom use cases – Utilize advanced prompt structures and formatting to meet intricate, project-specific requirements.

This guide explores how to tap into CrewAI’s prompts at a lower level, giving you fine-grained control over how agents think and interact.


## Understanding CrewAI’s Prompt System

Under the hood, CrewAI employs a modular prompt system that you can customize extensively:

### Agent templates – Govern each agent’s approach to their assigned role.

### Prompt slices – Control specialized behaviors such as tasks, tool usage, and output structure.

### Error handling – Direct how agents respond to failures, exceptions, or timeouts.

### Tool-specific prompts – Define detailed instructions for how tools are invoked or utilized.

## Understanding Default System Instructions

Production Transparency Issue: CrewAI automatically injects default instructions into your prompts that you might not be aware of. This section explains what’s happening under the hood and how to gain full control.

When you define an agent with role, goal, and backstory, CrewAI automatically adds additional system instructions that control formatting and behavior. Understanding these default injections is crucial for production systems where you need full prompt transparency.

## What CrewAI Automatically Injects

Based on your agent configuration, CrewAI adds different default instructions:
​
### For Agents Without Tools

"I MUST use these formats, my job depends on it!"
​

### For Agents With Tools

"IMPORTANT: Use the following format in your response:

Thought: you should always think about what to do
Action: the action to take, only one name of [tool_names]
Action Input: the input to the action, just a simple JSON object...
​

### For Structured Outputs (JSON/Pydantic)
"Ensure your final answer contains only the content in the following format: {output_format}
Ensure the final output does not include any code block markers like ```json or ```python."


## Instll necessary Libraries

In [ ]:
!pip install -q crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/

In [ ]:
import crewai
print(crewai.__version__)

1.15.17


# Set API Keys

In [ ]:
from google.colab import userdata
import os
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

## Import Dependencies

In [ ]:
from crewai import Agent, Task, Crew, LLM

## Create the LLM Object

In [ ]:
# OPENROUTER hosted LLMs
llm = LLM(
    model="openrouter/openai/gpt-oss-120b",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)


## Viewing the Complete System Prompt

To see exactly what prompt is being sent to your LLM, you can inspect the generated prompt:

# Define Agent and Task

In [ ]:
from crewai import Agent, Crew, Task
from crewai.utilities.prompts import Prompts

# Create your agent
agent = Agent(
    role="Data Analyst",
    goal="Analyze data and provide insights",
    backstory="You are an expert data analyst with 10 years of experience.",
    llm=llm,
    verbose=True
)

# Create a sample task
task = Task(
    description="Analyze the sales data and identify trends",
    expected_output="A detailed analysis with key insights and trends",
    agent=agent
)

## Define Crew

In [ ]:
# Create the prompt generator
prompt_generator = Prompts(
    agent=agent,
    has_tools=len(agent.tools) > 0,
    use_system_prompt=agent.use_system_prompt
)

# Generate and inspect the actual prompt
generated_prompt = prompt_generator.task_execution()

# Print the complete system prompt that will be sent to the LLM
if "system" in generated_prompt:
    print("=== SYSTEM PROMPT ===")
    print(generated_prompt["system"])
    print("\n=== USER PROMPT ===")
    print(generated_prompt["user"])
else:
    print("=== COMPLETE PROMPT ===")
    print(generated_prompt["prompt"])

# You can also see how the task description gets formatted
print("\n=== TASK CONTEXT ===")
print(f"Task Description: {task.description}")
print(f"Expected Output: {task.expected_output}")

=== SYSTEM PROMPT ===
You are Data Analyst. You are an expert data analyst with 10 years of experience.
Your personal goal is: Analyze data and provide insights

=== USER PROMPT ===

Current Task: {input}

Provide your complete response:

=== TASK CONTEXT ===
Task Description: Analyze the sales data and identify trends
Expected Output: A detailed analysis with key insights and trends


## Overriding Default Instructions

You have several options to gain full control over the prompts:


## Option 1: Custom Templates (Recommended)

In [ ]:
from crewai import Agent

# Define your own system template without default instructions
custom_system_template = """You are {role}. {backstory}
Your goal is: {goal}

Respond naturally and conversationally. Focus on providing helpful, accurate information."""

custom_prompt_template = """Task: {input}

Please complete this task thoughtfully."""

agent = Agent(
    role="Research Assistant",
    goal="Help users find accurate information",
    backstory="You are a helpful research assistant.",
    llm=llm,
    system_template=custom_system_template,
    prompt_template=custom_prompt_template,
    use_system_prompt=True  # Use separate system/user messages
)

In [ ]:
# Create the prompt generator
prompt_generator = Prompts(
    agent=agent,
    has_tools=len(agent.tools) > 0,
    use_system_prompt=agent.use_system_prompt
)

# Generate and inspect the actual prompt
generated_prompt = prompt_generator.task_execution()

# Print the complete system prompt that will be sent to the LLM
if "system" in generated_prompt:
    print("=== SYSTEM PROMPT ===")
    print(generated_prompt["system"])
    print("\n=== USER PROMPT ===")
    print(generated_prompt["user"])
else:
    print("=== COMPLETE PROMPT ===")
    print(generated_prompt["prompt"])

# You can also see how the task description gets formatted
print("\n=== TASK CONTEXT ===")
print(f"Task Description: {task.description}")
print(f"Expected Output: {task.expected_output}")

=== SYSTEM PROMPT ===
You are Research Assistant. You are a helpful research assistant.
Your personal goal is: Help users find accurate information

=== USER PROMPT ===

Current Task: {input}

Provide your complete response:

=== TASK CONTEXT ===
Task Description: Analyze the sales data and identify trends
Expected Output: A detailed analysis with key insights and trends


In [ ]:
print(generated_prompt["prompt"])


You are Research Assistant. You are a helpful research assistant.
Your personal goal is: Help users find accurate information
Current Task: {input}

Provide your complete response:


## Option 2: Custom Prompt File

Create a **custom_prompts.json** file to override specific prompt slices:

In [ ]:
{
  "slices": {
    "no_tools": "\nProvide your best answer in a natural, conversational way.",
    "tools": "\nYou have access to these tools: {tools}\n\nUse them when helpful, but respond naturally.",
    "formatted_task_instructions": "Format your response as: {output_format}"
  }
}

{'slices': {'no_tools': '\nProvide your best answer in a natural, conversational way.',
  'tools': '\nYou have access to these tools: {tools}\n\nUse them when helpful, but respond naturally.',
  'formatted_task_instructions': 'Format your response as: {output_format}'}}

Then use it in your crew:

In [ ]:
crew = Crew(
    agents=[agent],
    tasks=[task],
    prompt_file="custom_prompts.json",
    verbose=True
)